# Compact Endoscapes mAP — 3 Models

Focused chart: GPT-4.1-mini, Claude Haiku 4.5, Claude Opus 4.5 on Endoscapes.
Error bars shown automatically when multiple complete runs are available.

Methods: Direct, Direct+FS, CoT-first, CoT-first+FS, Rubric(w), Self-Rubric, Self-Rubric+FS.

In [ ]:
from __future__ import annotations

import json, re
from collections import defaultdict
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rubrics/cvs_rubrics_v3.json').is_file())
OUTPUT_ROOT = REPO_ROOT / 'outputs' / 'rubric_oracle'
RUBRICS_PATH = REPO_ROOT / 'rubrics' / 'cvs_rubrics_v3.json'
CRITS = ['c1', 'c2', 'c3']
COMMON_ONLY = True
RUN_SIMILARITY_THRESHOLD = 0.9
MIN_COMMON_RATIO = 0.7

ENDO_DIR = OUTPUT_ROOT / 'manifests' / 'test_dev'

# ── Rubric weights ──
rubrics_data = json.loads(RUBRICS_PATH.read_text())
rubric_items: Dict[str, Dict[str, object]] = {}
for criterion, block in rubrics_data.get('criteria', {}).items():
    for item in block.get('items', []):
        item_id = item.get('id')
        if item_id:
            rubric_items[item_id] = {'criterion': criterion,
                                     'weight': float(item.get('weight', 0.0))}

def status_to_bin(s):
    return 1 if s == 'yes' else 0

def unweighted_score_from_rubrics(pred_rubrics):
    """Unweighted mean: yes=1, uncertain/no=0, all items equal."""
    totals, counts = {}, {}
    for item_id, status in pred_rubrics.items():
        meta = rubric_items.get(item_id)
        if not meta:
            continue
        c = str(meta['criterion']).lower()
        totals[c] = totals.get(c, 0.0) + status_to_bin(status)
        counts[c] = counts.get(c, 0) + 1
    return {c: (totals.get(c, 0.0) / counts[c] if counts.get(c, 0) else float('nan'))
            for c in CRITS}

def weighted_score_from_rubrics(pred_rubrics):
    totals, weights = {}, {}
    for item_id, status in pred_rubrics.items():
        meta = rubric_items.get(item_id)
        if not meta:
            continue
        c = str(meta['criterion']).lower()
        w = float(meta.get('weight', 0.0))
        totals[c] = totals.get(c, 0.0) + w * status_to_bin(status)
        weights[c] = weights.get(c, 0.0) + w
    return {c: (totals.get(c, 0.0) / weights[c] if weights.get(c, 0) else float('nan'))
            for c in CRITS}

def weighted_score_partial(pred_rubrics):
    totals, weights = {}, {}
    for item_id, status in pred_rubrics.items():
        meta = rubric_items.get(item_id)
        if not meta:
            continue
        c = str(meta['criterion']).lower()
        w = float(meta.get('weight', 0.0))
        if status == 'yes':
            totals[c] = totals.get(c, 0.0) + w * 1.0
        elif status == 'uncertain':
            totals[c] = totals.get(c, 0.0) + w * 0.5
        weights[c] = weights.get(c, 0.0) + w
    return {c: (totals.get(c, 0.0) / weights[c] if weights.get(c, 0) else float('nan'))
            for c in CRITS}

def last2_product_score(pred_rubrics):
    def _val(status):
        if status == 'yes':
            return 1.0
        if status == 'uncertain':
            return 0.5
        return 0.0
    crit_items = {}
    for item_id, status in pred_rubrics.items():
        meta = rubric_items.get(item_id)
        if not meta:
            continue
        c = str(meta['criterion']).lower()
        crit_items.setdefault(c, []).append((item_id, status))
    result = {}
    for c in CRITS:
        items = crit_items.get(c, [])
        items.sort(key=lambda x: x[0])
        if len(items) < 3:
            result[c] = float('nan')
            continue
        last2_prod = _val(items[-2][1]) * _val(items[-1][1])
        prev_mean = sum(_val(s) for _, s in items[:-2]) / len(items[:-2])
        result[c] = last2_prod * prev_mean
    return result

RUBRIC_SCORE_FUNCS = {
    'unweighted': unweighted_score_from_rubrics,
    'weighted': weighted_score_from_rubrics,
    'weighted_partial': weighted_score_partial,
    'last2_product': last2_product_score,
}

def iter_jsonl(path):
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def safe_ap(y_true, y_score):
    if len(set(y_true)) < 2:
        return float('nan')
    return float(average_precision_score(y_true, y_score))

def compute_ap(y_true, y_score):
    y_true = np.asarray(y_true, dtype=float)
    y_score = np.asarray(y_score, dtype=float)
    mask = ~np.isnan(y_score)
    y_true, y_score = y_true[mask], y_score[mask]
    if y_true.size == 0:
        return float('nan')
    return safe_ap(y_true.tolist(), y_score.tolist())

def fmt_pct(mean, std):
    if mean != mean:
        return '--'
    if std > 0:
        return f'{mean * 100:.1f} +/- {std * 100:.1f}'
    return f'{mean * 100:.1f}'

def parse_avg(s):
    if not s or s.startswith('--'):
        return float('nan'), 0.0
    parts = s.split('+/-')
    return float(parts[0].strip()), (float(parts[1].strip()) if len(parts) > 1 else 0.0)

print('Setup complete. Rubric items:', len(rubric_items))

In [ ]:
FILENAME_RE = re.compile(
    r'^(?:endoscapes_val|cvs_challenge_sages_v1_val)__cvsrubricsv3__annv2__(.+)__seed13__([^/]+)\.jsonl$'
)
SUFFIX_RE = re.compile(r'(?:__\d{8}_\d{6}|_run\d{3})$')

def count_lines(path):
    with open(path) as f:
        return sum(1 for _ in f)

def discover_specs(root):
    if not root.exists():
        return []
    raw = []
    for entry in json.loads((REPO_ROOT / 'docs/prediction_manifest.json').read_text()):
        fpath = REPO_ROOT / entry['path']
        if '_old' in fpath.name:
            continue
        match = FILENAME_RE.match(fpath.name)
        if not match:
            continue
        suffix, model_raw = match.groups()
        label = suffix.strip('_')
        model = SUFFIX_RE.sub('', model_raw)
        raw.append({'model': model, 'experiment_label': label, 'path': fpath})

    groups: dict[tuple, list[Path]] = defaultdict(list)
    for s in raw:
        groups[(s['model'], s['experiment_label'])].append(s['path'])

    specs = []
    for (m, e), ps in sorted(groups.items()):
        ps = sorted(ps)
        if len(ps) > 1:
            lc = [count_lines(p) for p in ps]
            mn, mx = min(lc), max(lc)
            if mx > 0 and mn >= RUN_SIMILARITY_THRESHOLD * mx:
                kept = ps
            else:
                best_idx = lc.index(max(lc))
                kept = [ps[best_idx]]
        else:
            kept = ps
        specs.append({'model': m, 'experiment_label': e, 'paths': kept,
                      'n_original': len(ps), 'n_kept': len(kept)})
    return specs

def collect_all_records(specs, dataset_tag):
    records = []
    for spec in specs:
        model, exp = spec['model'], spec['experiment_label']
        copy_counter = defaultdict(int)
        for fpath in spec['paths']:
            if not fpath.exists():
                continue
            for row in iter_jsonl(fpath):
                vid, fid = row.get('video_id', ''), row.get('frame_id', '')
                row_gt = row.get('gt')
                if not isinstance(row_gt, dict):
                    continue
                gt = {}
                for k in CRITS:
                    v = row_gt.get(k)
                    if v is not None:
                        gt[k] = 1 if float(v) > 0.5 else 0
                if not gt:
                    continue
                key = (vid, fid)
                ci = copy_counter[key]
                copy_counter[key] += 1
                fk = f'{vid}/{fid}'
                pred = row.get('pred', {}) or {}
                rubrics = row.get('rubric_labels_stage1', {}) or {}
                for c in CRITS:
                    gb = gt.get(c)
                    if gb is None:
                        continue
                    base = {'dataset': dataset_tag, 'model': model,
                            'experiment_label': exp, 'criterion': c,
                            'y_true': int(gb), 'file_name': fk,
                            'video_id': vid, 'frame_id': fid, 'copy_idx': ci}
                    ds = float(pred.get(c, float('nan')))
                    if ds == ds:
                        records.append({**base, 'score_type': 'direct', 'y_score': ds})
                    if rubrics:
                        for stype, func in RUBRIC_SCORE_FUNCS.items():
                            sc = func(rubrics)
                            sv = float(sc.get(c, float('nan')))
                            if sv == sv:
                                records.append({**base, 'score_type': stype, 'y_score': sv})
    return pd.DataFrame(records)

endo_specs = discover_specs(ENDO_DIR)
print(f'Endoscapes: {len(endo_specs)} configs')

# Show run info for models of interest
MODELS_OF_INTEREST = {'gpt-4.1-mini', 'claude-haiku-4-5-20251001', 'claude-opus-4-5-20251101'}
for s in endo_specs:
    if s['model'] in MODELS_OF_INTEREST:
        e = s['experiment_label']
        is_nofs = not any(x in e for x in ['fsv', 'fsd'])
        is_fsv3 = 'fsv3' in e and 'fsd' not in e
        is_sr = 'sr_guided' in e
        if is_nofs or is_fsv3 or is_sr:
            print(f"  {s['model']:40s}  {e:70s}  {s['n_kept']}/{s['n_original']} runs")

df_endo = collect_all_records(endo_specs, 'endoscapes')
print(f'\nEndoscapes records: {len(df_endo)}')


In [ ]:
def compute_method_stats(df, exp_label, model, score_type='direct'):
    mask = ((df['model'] == model) & (df['experiment_label'] == exp_label) &
            (df['score_type'] == score_type))
    sub = df[mask]
    nan_stat = (float('nan'), float('nan'))
    if sub.empty:
        return {c: nan_stat for c in CRITS}, nan_stat, []

    n_copies = sub['copy_idx'].max() + 1
    if n_copies > 1:
        per_copy_aps = {c: [] for c in CRITS}
        per_copy_n = []
        for ci in range(n_copies):
            cdf = sub[sub['copy_idx'] == ci]
            per_copy_n.append(cdf['file_name'].nunique())
            for c in CRITS:
                cs = cdf[cdf['criterion'] == c]
                per_copy_aps[c].append(compute_ap(cs['y_true'].values, cs['y_score'].values))
        crit_stats = {}
        for c in CRITS:
            valid = [a for a in per_copy_aps[c] if a == a]
            m = float(np.mean(valid)) if valid else float('nan')
            s = float(np.std(valid, ddof=1)) if len(valid) > 1 else 0.0
            crit_stats[c] = (m, s)
        per_copy_avg = [float(np.nanmean([per_copy_aps[c][ci] for c in CRITS]))
                        for ci in range(n_copies)]
        valid_avg = [a for a in per_copy_avg if a == a]
        avg_m = float(np.mean(valid_avg)) if valid_avg else float('nan')
        avg_s = float(np.std(valid_avg, ddof=1)) if len(valid_avg) > 1 else 0.0
        return crit_stats, (avg_m, avg_s), per_copy_n
    else:
        n_frames = [sub['file_name'].nunique()]
        crit_stats = {}
        for c in CRITS:
            cs = sub[sub['criterion'] == c]
            crit_stats[c] = (compute_ap(cs['y_true'].values, cs['y_score'].values), 0.0)
        avg_ap = float(np.nanmean([crit_stats[c][0] for c in CRITS]))
        return crit_stats, (avg_ap, 0.0), n_frames


def filter_common_frames(df, method_tuples, model, verbose=True):
    if not COMMON_ONLY:
        return df
    all_exps = set()
    for _, nofs, fs, _ in method_tuples:
        if nofs:
            all_exps.add(nofs)
        if fs:
            all_exps.add(fs)
    exp_frame_sets = {}
    for exp in all_exps:
        mask = ((df['model'] == model) & (df['experiment_label'] == exp) &
                (df['score_type'] == 'direct'))
        sub = df[mask]
        if sub.empty:
            continue
        # Use union of frames across all copies for common-set computation
        exp_frame_sets[exp] = set(sub['file_name'].unique())
    if not exp_frame_sets:
        return df
    max_frames = max(len(fs) for fs in exp_frame_sets.values())
    kept_sets = []
    for exp, fs in exp_frame_sets.items():
        if len(fs) < MIN_COMMON_RATIO * max_frames:
            if verbose:
                print(f'  EXCLUDED from common: {exp} ({len(fs)} frames, max={max_frames})')
        else:
            kept_sets.append(fs)
    if not kept_sets:
        return df
    common = kept_sets[0]
    for fs in kept_sets[1:]:
        common &= fs
    rel = (df['model'] == model)
    # Keep ALL copies for common frames so multi-run mean/std can be computed
    keep = df['file_name'].isin(common)
    df_out = df[~rel | (rel & keep)].copy()
    if verbose:
        print(f'  COMMON: {len(common)} frames (keeping all copies)')
    return df_out


def parse_n_values(s):
    if s == '--' or not s:
        return []
    return [int(x) for x in s.split('/')]

def should_blank(n_common_str, n_total_str):
    nc = parse_n_values(n_common_str)
    nt = parse_n_values(n_total_str)
    if not nc or not nt:
        return False
    avg_c = sum(nc) / len(nc)
    avg_t = sum(nt) / len(nt)
    return avg_t > 0 and avg_c < MIN_COMMON_RATIO * avg_t

def should_blank_vs_ref(n_total_str, ref_frames):
    nt = parse_n_values(n_total_str)
    if not nt or ref_frames <= 0:
        return False
    avg_t = sum(nt) / len(nt)
    return avg_t < MIN_COMMON_RATIO * ref_frames


# ── Models + methods ──
ALL_MODELS = [
    ('gpt-4.1-mini',              'GPT-4.1-mini'),
    ('claude-haiku-4-5-20251001', 'Claude Haiku 4.5'),
    ('claude-opus-4-5-20251101',  'Claude Opus 4.5'),
]

split = 'test_dev'

def make_method_tuples():
    rubric_exp = f'predicted_only__{split}__critdef__fsv3__fs4'
    return [
        ('Direct',
         f'direct__preset-direct__{split}__critdef',
         f'direct__preset-direct__{split}__critdef__fsv3__fs4',
         'direct'),
        ('CoT-first',
         f'direct__preset-direct__{split}__critdef__ratdf',
         f'direct__preset-direct__{split}__critdef__ratdf__fsv3__fs4',
         'direct'),
        ('Rubric',         None, rubric_exp, 'direct'),
        ('Rubric(w)',      None, rubric_exp, 'weighted'),
        ('Rubric(p)',      None, rubric_exp, 'weighted_partial'),
        ('Rubric(l2)',     None, rubric_exp, 'last2_product'),
        ('Self-Rubric',
         f'self_rubric__{split}__critdef__rat1__s1short',
         f'self_rubric__{split}__critdef__rat1__s1short__fsv3__fs4',
         'direct'),
    ]

method_tuples = make_method_tuples()

# Compute reference frame count
ref_frames = 0
for model_id, _ in ALL_MODELS:
    sub = df_endo[(df_endo['model'] == model_id) & (df_endo['score_type'] == 'direct')]
    if not sub.empty:
        n = sub[sub['copy_idx'] == 0]['file_name'].nunique()
        ref_frames = max(ref_frames, n)
print(f'Reference frames for Endoscapes: {ref_frames}')

# ── Evaluate ──
all_results = {}
all_df_f = {}  # save filtered dfs for LaTeX table generation
for model_id, model_name in ALL_MODELS:
    sub = df_endo[df_endo['model'] == model_id]
    if sub.empty:
        print(f'{model_name}: No data')
        all_results[(model_id, 'Endoscapes')] = []
        continue
    print(f'\n--- {model_name} ---')
    df_f = filter_common_frames(df_endo, method_tuples, model_id)
    all_df_f[model_id] = df_f
    nan_stat = (float('nan'), float('nan'))
    rows = []
    for mname, nofs_exp, fs_exp, stype in method_tuples:
        row = {'Method': mname}
        # no FS
        if nofs_exp:
            nc_ap, na_ap, nn = compute_method_stats(df_f, nofs_exp, model_id, stype)
            _, _, nn_total = compute_method_stats(df_endo, nofs_exp, model_id, stype)
        else:
            nc_ap = {c: nan_stat for c in CRITS}
            na_ap = nan_stat; nn = []; nn_total = []
        row['Avg AP (no FS)'] = fmt_pct(*na_ap)
        row['n_common (no FS)'] = '/'.join(str(x) for x in nn) if nn else '--'
        row['n_total (no FS)'] = '/'.join(str(x) for x in nn_total) if nn_total else '--'
        if (should_blank(row['n_common (no FS)'], row['n_total (no FS)']) or
                should_blank_vs_ref(row['n_total (no FS)'], ref_frames)):
            row['Avg AP (no FS)'] = '--*'
        # fsv3
        if fs_exp:
            fc_ap, fa_ap, fn = compute_method_stats(df_f, fs_exp, model_id, stype)
            _, _, fn_total = compute_method_stats(df_endo, fs_exp, model_id, stype)
        else:
            fc_ap = {c: nan_stat for c in CRITS}
            fa_ap = nan_stat; fn = []; fn_total = []
        row['Avg AP (fsv3)'] = fmt_pct(*fa_ap)
        row['n_common (fsv3)'] = '/'.join(str(x) for x in fn) if fn else '--'
        row['n_total (fsv3)'] = '/'.join(str(x) for x in fn_total) if fn_total else '--'
        if (should_blank(row['n_common (fsv3)'], row['n_total (fsv3)']) or
                should_blank_vs_ref(row['n_total (fsv3)'], ref_frames)):
            row['Avg AP (fsv3)'] = '--*'
        rows.append(row)
    all_results[(model_id, 'Endoscapes')] = rows
    df_rows = pd.DataFrame(rows)
    display(df_rows[['Method', 'Avg AP (no FS)', 'Avg AP (fsv3)',
                     'n_common (fsv3)', 'n_total (fsv3)']])

print('\nEvaluation complete.')

In [ ]:
# ── LaTeX table: per-criterion mAP with best bolded ──

# Methods for the table: (latex_display_name, source_method_in_method_tuples, variant)
TABLE_METHODS = [
    ('Direct',            'Direct',      'no FS'),
    ('Direct+FS',         'Direct',      'fsv3'),
    ('CoT',               'CoT-first',   'no FS'),
    ('CoT+FS',            'CoT-first',   'fsv3'),
    ('SubQ',              'Self-Rubric', 'no FS'),
    ('SubQ+FS',           'Self-Rubric', 'fsv3'),
    ('OURS',              'Rubric(w)',   'fsv3'),
]

# Build experiment lookup from method_tuples
exp_lookup = {}
for mname, nofs_exp, fs_exp, stype in method_tuples:
    exp_lookup[mname] = {'no FS': nofs_exp, 'fsv3': fs_exp, 'score_type': stype}

# Collect per-criterion stats for each model x method
# table_stats[model_id][display_name] = {'c1': (m,s), 'c2': (m,s), 'c3': (m,s), 'avg': (m,s)}
columns = ['c1', 'c2', 'c3', 'avg']
table_stats = {}
for model_id, model_name in ALL_MODELS:
    df_f = all_df_f.get(model_id)
    if df_f is None:
        table_stats[model_id] = {}
        continue
    model_stats = {}
    for display_name, source_method, variant in TABLE_METHODS:
        info = exp_lookup.get(source_method, {})
        exp = info.get(variant)
        stype = info.get('score_type', 'direct')
        if exp:
            crit_stats, avg_stat, _ = compute_method_stats(df_f, exp, model_id, stype)
        else:
            nan_stat = (float('nan'), float('nan'))
            crit_stats = {c: nan_stat for c in CRITS}
            avg_stat = nan_stat
        model_stats[display_name] = {**crit_stats, 'avg': avg_stat}
    table_stats[model_id] = model_stats

# Find best (highest mean) per column per model
best_per_model = {}
for model_id in [m[0] for m in ALL_MODELS]:
    best = {}
    for col in columns:
        best_val = -float('inf')
        for dn, _, _ in TABLE_METHODS:
            if dn not in table_stats.get(model_id, {}):
                continue
            mean, _ = table_stats[model_id][dn][col]
            if mean == mean and mean > best_val:
                best_val = mean
        best[col] = best_val if best_val > -float('inf') else float('nan')
    best_per_model[model_id] = best

def fmt_latex_cell(mean, std, is_best=False):
    if mean != mean:
        return '--'
    if std > 0:
        val = f'{mean*100:.1f}$_{{\\pm {std*100:.1f}}}$'
    else:
        val = f'{mean*100:.1f}'
    if is_best:
        val = f'\\textbf{{{val}}}'
    return val

# ── Model name mapping for makecell ──
MODEL_MAKECELL = {
    'GPT-4.1-mini':    r'\makecell{\textbf{GPT-4.1}\\\textbf{mini}}',
    'Claude Haiku 4.5': r'\makecell{\textbf{Claude}\\\textbf{Haiku 4.5}}',
    'Claude Opus 4.5':  r'\makecell{\textbf{Claude}\\\textbf{Opus 4.5}}',
}

# ── Generate LaTeX ──
n_methods = len(TABLE_METHODS)
lines = []
lines.append(r'\begin{table}[t]')
lines.append(r'\caption{\textbf{Frame-level CVS assessment performance (mAP).} Cells show mean mAP (\%) across runs; $\pm$ indicates std across runs. Best results per model bolded.}')
lines.append(r'\label{tab:frame_map_main}')
lines.append(r'\centering')
lines.append(r'\setlength{\tabcolsep}{6pt}')
lines.append(r'\renewcommand{\arraystretch}{1.05}')
lines.append(r'\scriptsize')
lines.append(r'\begin{tabular}{l l cccc}')
lines.append(r'\toprule')
lines.append(r'\textbf{Model} & \textbf{Method} & \textbf{C1} & \textbf{C2} & \textbf{C3} & \textbf{Avg} \\')
lines.append(r'\midrule')
lines.append('')

for idx, (model_id, model_name) in enumerate(ALL_MODELS):
    if idx > 0:
        lines.append(r'\midrule')
        lines.append('')

    makecell = MODEL_MAKECELL.get(model_name, r'\textbf{' + model_name + '}')
    model_tex = f'\\multirow{{{n_methods}}}{{*}}{{{makecell}}}'
    best = best_per_model.get(model_id, {})

    for j, (display_name, source_method, variant) in enumerate(TABLE_METHODS):
        stats = table_stats.get(model_id, {}).get(display_name, {})
        if not stats:
            cells = ['--'] * 4
        else:
            cells = []
            for col in columns:
                mean, std = stats[col]
                is_best = (mean == mean and best.get(col, None) == best.get(col, None)
                           and abs(mean - best.get(col, float('nan'))) < 1e-9)
                cells.append(fmt_latex_cell(mean, std, is_best))

        # Method name formatting
        if display_name == 'OURS':
            method_tex = r'\textbf{\OURS{}}'
        else:
            method_tex = display_name

        if j == 0:
            row_prefix = model_tex
        else:
            row_prefix = ''

        line = f'{row_prefix}\n& {method_tex} \n& {cells[0]} \n& {cells[1]} \n& {cells[2]} \n& {cells[3]} \\\\'
        lines.append(line)

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(r'\end{table}')

latex_str = '\n'.join(lines)
print(latex_str)

In [ ]:
# ── Ablation table: GPT-4.1-mini Rubric variants ──
# Three rows:
#   1. \OURS{} (no FS)          → predicted_only__test_dev__critdef,          score_type='direct'
#   2. \OURS{} (LLM agg.)      → predicted_only__test_dev__critdef__fsv3__fs4, score_type='direct'
#   3. \OURS{} (Weighted) (ours)→ predicted_only__test_dev__critdef__fsv3__fs4, score_type='weighted'

ABL_MODEL = 'gpt-4.1-mini'
ABL_MODEL_NAME = 'GPT-4.1-mini'

# (internal_key, experiment, score_type, latex_method_name)
ABL_ROWS = [
    ('noFS', 'predicted_only__test_dev__critdef', 'weighted', r'No FS (weighted)'),
    ('noFS_llm', 'predicted_only__test_dev__critdef', 'direct', r'No FS (LLM; PDF row)'),
    ('llmagg',  'predicted_only__test_dev__critdef__fsv3__fs4',   'direct',   r'\OURS{} (LLM agg.)'),
    ('ours',    'predicted_only__test_dev__critdef__fsv3__fs4',   'weighted', r'\textbf{\OURS{} (Weighted)}  (ours)'),
]

# For the ablation we filter to frames common across just these experiments for GPT
abl_exps = list(set(exp for _, exp, _, _ in ABL_ROWS))
abl_frame_sets = {}
for exp in abl_exps:
    mask = ((df_endo['model'] == ABL_MODEL) & (df_endo['experiment_label'] == exp) &
            (df_endo['score_type'] == 'direct'))
    sub = df_endo[mask]
    if not sub.empty:
        abl_frame_sets[exp] = set(sub['file_name'].unique())
        print(f'  {exp}: {len(abl_frame_sets[exp])} frames')

if abl_frame_sets:
    abl_common = set.intersection(*abl_frame_sets.values())
    print(f'  Common frames for ablation: {len(abl_common)}')
else:
    abl_common = set()

# Filter df_endo to common frames for GPT ablation
abl_mask = (df_endo['model'] == ABL_MODEL) & (df_endo['file_name'].isin(abl_common))
df_abl = df_endo[abl_mask].copy()

# Compute stats
abl_stats = {}
for key, exp, stype, latex_name in ABL_ROWS:
    crit_stats, avg_stat, n_frames = compute_method_stats(df_abl, exp, ABL_MODEL, stype)
    abl_stats[key] = {**crit_stats, 'avg': avg_stat}
    n_str = '/'.join(str(x) for x in n_frames) if n_frames else '--'
    print(f'  {latex_name:45s}  avg={fmt_pct(*avg_stat):20s}  n={n_str}')

# Find best per column
abl_best = {}
for col in columns:
    best_val = -float('inf')
    for key, _, _, _ in ABL_ROWS:
        mean, _ = abl_stats.get(key, {}).get(col, (float('nan'), 0.0))
        if mean == mean and mean > best_val:
            best_val = mean
    abl_best[col] = best_val if best_val > -float('inf') else float('nan')

# Generate LaTeX
abl_lines = []
abl_lines.append(r'\begin{table}[t]')
abl_lines.append(r'\caption{\textbf{Ablation: Rubric aggregation variants (' + ABL_MODEL_NAME + r').} '
                  r'Impact of few-shot examples and aggregation strategy on mAP (\%).}')
abl_lines.append(r'\label{tab:ablation_rubric}')
abl_lines.append(r'\centering')
abl_lines.append(r'\setlength{\tabcolsep}{6pt}')
abl_lines.append(r'\renewcommand{\arraystretch}{1.05}')
abl_lines.append(r'\scriptsize')
abl_lines.append(r'\begin{tabular}{l cccc}')
abl_lines.append(r'\toprule')
abl_lines.append(r'\textbf{Method} & \textbf{C1} & \textbf{C2} & \textbf{C3} & \textbf{Avg} \\')
abl_lines.append(r'\midrule')

for key, _, _, latex_name in ABL_ROWS:
    stats = abl_stats.get(key, {})
    cells = []
    for col in columns:
        mean, std = stats.get(col, (float('nan'), 0.0))
        is_best = (mean == mean and abs(mean - abl_best.get(col, float('nan'))) < 1e-9)
        cells.append(fmt_latex_cell(mean, std, is_best))

    line = f'{latex_name} & {cells[0]} & {cells[1]} & {cells[2]} & {cells[3]} \\\\'
    abl_lines.append(line)

abl_lines.append(r'\bottomrule')
abl_lines.append(r'\end{tabular}')
abl_lines.append(r'\end{table}')

abl_latex = '\n'.join(abl_lines)
print('\n' + abl_latex)